# Comparing Scenarios on `premise` Data

`TimexLCASettings` holds everything one calculation needs - the demand, the
method, the background selection, and every timeline/LCI/LCIA option - so one
object is also the record of what was run. `TimexLCA(...).run()`
executes it, `run()` can be called again with overrides, and
`TimexLCA.compare()` runs a list of them into one table.

We follow that through on a real prospective background, from an empty
project to a scenario comparison: build ecoinvent 3.12 (cutoff) plus REMIND-EU databases for 2020-2040 with
[premise](https://github.com/polca/premise), point a small electric-vehicle
foreground at them, and compare EV impacts 1) depending on the year it is
bought and 2) depending on the scenario.

## Setting up the databases

We start totally from scratch, in am empty project:

In [1]:
import bw2data as bd

bd.projects.set_current("my_new_timex_project")
list(bd.databases)

[]

We can download the ecoinvent database and create scenario-copies of it with `premise` through our `ensure_scenario_databases` helper function. Running this requires a premise key (to be asked from `premise` maintainers) and valid ecoinvent credentials. These can be set as environment variables (`PREMISE_KEY`, `ECOINVENT_USERNAME`, `ECOINVENT_PASSWORD`) or passed to the function directly.

In [2]:
from bw_timex import ensure_scenario_databases

database_dates = ensure_scenario_databases(
    {
        "iam_model": "remind-eu",
        "pathway": "SSP2-PkBudg650",
        "system_model": "cutoff",
        "ecoinvent_version": "3.12",
        "years": [2020, 2030, 2040],
    },
    # premise_key="dummy_premise_decryption_key",
    # ecoinvent_credentials=("dummy_user", "dummy_password"),
)

2026-08-26 08:21:05.283 | INFO     | bw_timex.scenario_builder:_resolve_source_database:289 - No database 'ecoinvent-3.12-cutoff' in this project. Importing ecoinvent 3.12 (cutoff) first; this takes a while and needs an ecoinvent licence.


08:21:05+0200 [warning  ] Can't import `SimaProBlockCSVImporter` - please install `bw2io` with `pip install bw2io[multifunctional]` or install `multifunctional` and `bw_simapro_csv` manually.
Applying strategy: normalize_units
Applying strategy: drop_unspecified_subcategories
Applying strategy: ensure_categories_are_tuples
Applied 3 strategies in 0.00 seconds
Graph statistics for `ecoinvent-3.12-biosphere` importer:
9850 graph nodes:
	emission: 9477
	natural resource: 353
	inventory indicator: 15
	economic: 5
0 graph edges:
0 edges to the following databases:
0 unique unlinked edges (0 total):




100%|██████████| 9850/9850 [00:00<00:00, 55672.35it/s]

08:21:12+0200 [info     ] Vacuuming database            


Created database: ecoinvent-3.12-biosphere
Extracting XML data from 26533 datasets
08:21:16+0200 [warning  ] Can't import `SimaProBlockCSVImporter` - please install `bw2io` with `pip install bw2io[multifunctional]` or install `multifunctional` and `bw_simapro_csv` manually.
08:21:16+0200 [warning  ] Can't import `SimaProBlockCSVImporter` - please install `bw2io` with `pip install bw2io[multifunctional]` or install `multifunctional` and `bw_simapro_csv` manually.
08:21:16+0200 [warning  ] Can't import `SimaProBlockCSVImporter` - please install `bw2io` with `pip install bw2io[multifunctional]` or install `multifunctional` and `bw_simapro_csv` manually.
08:21:16+0200 [warning  ] Can't import `SimaProBlockCSVImporter` - please install `bw2io` with `pip install bw2io[multifunctional]` or install `multifunctional` and `bw_simapro_csv` manually.
08:21:16+0200 [warning  ] Can't import `SimaProBlockCSVImporter` - please install `bw2io` with `pip install bw2io[multifunctional]` or install `multi

...

Brightway database written: ei_cutoff_3.12_remind-eu_SSP2-PkBudg650_2020
Running core export checks...
Minor anomalies found: check the change report.


Brightway database written: ei_cutoff_3.12_remind-eu_SSP2-PkBudg650_2030
Running core export checks...
Minor anomalies found: check the change report.


Brightway database written: ei_cutoff_3.12_remind-eu_SSP2-PkBudg650_2040
Generate scenario report.
Report saved under /Users/timodiepers/Documents/Coding/bw_timex/notebooks/advanced/export/scenario_report.
Generate change report.


2026-08-26 08:35:23.606 | INFO     | bw_timex.scenario_builder:ensure_scenario_databases:424 - Built 3 background database(s).


Report saved under /Users/timodiepers/Documents/Coding/bw_timex/notebooks/advanced/export/change reports/.


We now find the base ecoinvent database as well as the three time-specific premise databases in our project:

In [3]:
list(bd.databases)

['ecoinvent-3.12-biosphere',
 'ecoinvent-3.12-cutoff',
 'ei_cutoff_3.12_remind-eu_SSP2-PkBudg650_2020',
 'ei_cutoff_3.12_remind-eu_SSP2-PkBudg650_2030',
 'ei_cutoff_3.12_remind-eu_SSP2-PkBudg650_2040']

Next, we add a foreground database. We use the same electric vehicle example as in [this teaching notebook](../teaching/ev_walktrough_premise.ipynb), which we can set up like this:

In [4]:
from bw_timex import create_electric_vehicle_example

create_electric_vehicle_example(
    background_database_name="ei_cutoff_3.12_remind-eu_SSP2-PkBudg650_2020"
)

2026-08-26 08:35:25.305 | INFO     | bw_timex.utils:add_temporal_distribution_to_exchange:670 - Added temporal distribution to exchange Exchange: 20000.0 kilowatt hour 'market group for electricity, low voltage' (kilowatt hour, GLO, None) to 'driving an electric vehicle' (pkm over ev lifetime, GLO, None).
2026-08-26 08:35:25.309 | INFO     | bw_timex.utils:add_temporal_distribution_to_exchange:670 - Added temporal distribution to exchange Exchange: -280 kilogram 'market for used Li-ion battery' (kilogram, GLO, None) to 'driving an electric vehicle' (pkm over ev lifetime, GLO, None).
2026-08-26 08:35:25.310 | INFO     | bw_timex.utils:create_electric_vehicle_example:1271 - Created electric vehicle example in database 'foreground'.


## Calculating a `TimexLCA` using `TimexLCASetting``

In [5]:
from bw_timex import TimexLCA, TimexLCASettings

settings = TimexLCASettings(
    demand = {("foreground", "driving"): 1},
    method = ("ecoinvent-3.12", "EF v3.1", "climate change", "global warming potential (GWP100)"),
    scenario={
        "iam_model": "remind-eu",
        "pathway": "SSP2-PkBudg650",
    },
    timeline={ # Settings normally going into .build_timeline()
        "starting_datetime": "2028-01-01",
        "graph_traversal": "bfs",
    },
    lci={}, # Settings normally going into .lci()
    lcia={  # Settings normally going into .static_lcia() or .dynamic_lcia()
        "metric": "GWP",
        "time_horizon": 100,
    },
)

In [6]:
tlca = TimexLCA(settings).run()

print("Time-explicit impact:", tlca.dynamic_score, bd.methods[settings.method]["unit"])

2026-08-26 08:35:25.320 | INFO     | bw_timex.timex_lca:__init__:444 - Initializing TimexLCA object...
2026-08-26 08:35:25.342 | INFO     | bw_timex.timex_lca:__init__:533 - Calculating base LCA...
2026-08-26 08:35:25.353 | INFO     | bw_timex.timex_lca:clean_databases:2475 - Reprocessing 1 modified database(s) before calculating: foreground. This can take a while for large databases.
2026-08-26 08:35:25.374 | INFO     | bw_timex.timex_lca:clean_databases:2481 - Done reprocessing modified databases.
/Users/timodiepers/Documents/Coding/bw_timex/.venv/lib/python3.12/site-packages/scikits/umfpack/umfpack.py:737: UmfpackWarning: (almost) singular matrix! (estimated cond. number: 8.06e+12)
  warnings.warn(msg, UmfpackWarning)
2026-08-26 08:35:26.402 | INFO     | bw_timex.timex_lca:__init__:588 - Collecting node infos...
2026-08-26 08:35:26.446 | INFO     | bw_timex.timex_lca:__init__:603 - Loading node metadata from 4 database(s)...
2026-08-26 08:35:29.312 | INFO     | bw_timex.timex_lca:__

Time-explicit impact: 27329.168393728134 kg CO2-Eq


Another run using the same `TimexLCA` object, overwriting the time horizon setting:

In [7]:
tlca.run(time_horizon=20) # same as tlca.run(lcia={"time_horizon": 20})
print("Time-explicit impact with time horizon 20:", tlca.dynamic_score, bd.methods[settings.method]["unit"])

2026-08-26 08:35:58.569 | INFO     | bw_timex.timex_lca:run:812 - Starting TimexLCA.run() pipeline...
2026-08-26 08:35:58.570 | INFO     | bw_timex.timex_lca:run:819 - Step 1/4: Building timeline...
2026-08-26 08:35:58.573 | INFO     | bw_timex.timex_lca:run:834 - Step 2/4: Calculating LCI...
2026-08-26 08:35:59.049 | INFO     | bw_timex.timex_lca:lci:1440 - Expanding matrices...
2026-08-26 08:35:59.062 | INFO     | bw_timex.timex_lca:lci:1459 - Calculating dynamic inventory...
2026-08-26 08:36:01.198 | INFO     | bw_timex.timex_lca:run:843 - Step 3/4: Calculating static LCIA...
2026-08-26 08:36:01.202 | INFO     | bw_timex.timex_lca:run:862 - Step 4/4: Calculating dynamic LCIA...
2026-08-26 08:36:01.219 | INFO     | dynamic_characterization.dynamic_characterization:characterize:135 - No custom dynamic characterization functions provided. Using default dynamic             characterization functions. The flows that are characterized are based on the selection                of the initi

Time-explicit impact with time horizon 20: 31108.70858992546 kg CO2-Eq


## Several calculations: `compare()`

To compare different settings even more easily, use the `compare()` function, which takes a list of settings. It returns a `ComparisonResult`
whose `summary` holds one row each - the scores next to every setting that
produced them, so the table is its own record of what was run.

### Compare different purchasing years

In [8]:
from dataclasses import replace

comparison = TimexLCA.compare(
    [
        replace(settings, starting_datetime=f"{year}-06-01", label=f"bought {year}")
        for year in (2020, 2025, 2030)
    ]
)

2026-08-26 08:36:01.299 | INFO     | bw_timex.timex_lca:__init__:444 - Initializing TimexLCA object...
2026-08-26 08:36:01.301 | INFO     | bw_timex.timex_lca:__init__:533 - Calculating base LCA...
/Users/timodiepers/Documents/Coding/bw_timex/.venv/lib/python3.12/site-packages/scikits/umfpack/umfpack.py:737: UmfpackWarning: (almost) singular matrix! (estimated cond. number: 8.06e+12)
  warnings.warn(msg, UmfpackWarning)
2026-08-26 08:36:02.271 | INFO     | bw_timex.timex_lca:__init__:588 - Collecting node infos...
2026-08-26 08:36:02.327 | INFO     | bw_timex.timex_lca:__init__:603 - Loading node metadata from 4 database(s)...
2026-08-26 08:36:02.430 | INFO     | bw_timex.timex_lca:__init__:648 - TimexLCA initialized.
2026-08-26 08:36:02.431 | INFO     | bw_timex.timex_lca:compare:1022 - Comparison 1/3: bought 2020
2026-08-26 08:36:02.431 | INFO     | bw_timex.timex_lca:run:812 - Starting TimexLCA.run() pipeline...
2026-08-26 08:36:02.432 | INFO     | bw_timex.timex_lca:run:819 - Step 

In [9]:
comparison.summary[["label", "base_score", "dynamic_score", "runtime_s"]]

,label,base_score,dynamic_score,runtime_s
0,bought 2020,23029.209074,38710.383983,15.966528
1,bought 2025,23029.209074,31377.740939,5.043378
2,bought 2030,23029.209074,24105.658666,4.676812


Two options worth knowing for `compare()`: `keep_objects=True` keeps each `TimexLCA`
in `ComparisonResult.objects`, to dig into one result's timeline or dynamic
inventory afterwards; `on_error="record"` puts a failure in the row's `error`
column and carries on, instead of aborting a long unattended sweep.

### Compare different scenarios

Instead of comparing different purchase years, we could also compare different *scenarios*. Give
each settings object a different `scenario={...}` (say `SSP2-PkBudg650` against
`SSP2-NPi`).

> Note how the SSP-NPi scenario does not exist in our project at this time. By setting `create_missing=True`, the respective premise scenario databases are created on the fly (internally using the `ensure_scenario_databases` function shown above), without any futher work.

In [10]:
from dataclasses import replace

comparison = TimexLCA.compare(
    [
        replace(
            settings,
            scenario={
                "pathway": pathway,
                "iam_model": "remind-eu",
                "ecoinvent_version": "3.12",
                "system_model": "cutoff",
                "years": [2020, 2030, 2040],
            },
            create_missing=True,
            label=f"Pathway {pathway}",
        )
        for pathway in ("SSP2-PkBudg650", "SSP2-PkBudg1000")
    ]
)

2026-08-26 08:36:28.226 | INFO     | bw_timex.timex_lca:__init__:444 - Initializing TimexLCA object...
2026-08-26 08:36:28.232 | INFO     | bw_timex.scenario_builder:ensure_scenario_databases:366 - All 3 requested background vintage(s) already exist in this project. Nothing to build.
2026-08-26 08:36:28.235 | INFO     | bw_timex.timex_lca:__init__:533 - Calculating base LCA...
/Users/timodiepers/Documents/Coding/bw_timex/.venv/lib/python3.12/site-packages/scikits/umfpack/umfpack.py:737: UmfpackWarning: (almost) singular matrix! (estimated cond. number: 8.06e+12)
  warnings.warn(msg, UmfpackWarning)
2026-08-26 08:36:29.249 | INFO     | bw_timex.timex_lca:__init__:588 - Collecting node infos...
2026-08-26 08:36:29.294 | INFO     | bw_timex.timex_lca:__init__:603 - Loading node metadata from 4 database(s)...
2026-08-26 08:36:29.358 | INFO     | bw_timex.timex_lca:__init__:648 - TimexLCA initialized.
2026-08-26 08:36:29.359 | INFO     | bw_timex.timex_lca:compare:1022 - Comparison 1/2: Pat

premise v.(2, 4, 9, 2)
+------------------------------------------------------------------+
| Warning                                                          |
+------------------------------------------------------------------+
| Because some of the scenarios can yield LCI databases            |
| containing net negative emission technologies (NET),             |
| it is advised to account for biogenic CO2 flows when calculating |
| Global Warming potential indicators.                             |
| `premise_gwp` provides characterization factors for such flows.  |
| It also provides factors for hydrogen emissions to air.          |
|                                                                  |
| Within your Brightway project:                                   |
| from premise_gwp import add_premise_gwp                          |
| add_premise_gwp()                                                |
+------------------------------------------------------------------+
+----------

Processing scenarios for all sectors: 100%|█| 3/3 [06:48<00:00, 136.21


Done!

Write new database(s) to Brightway.
Running core export checks...
Minor anomalies found: check the change report.


Brightway database written: ei_cutoff_3.12_remind-eu_SSP2-PkBudg1000_2020
Running core export checks...
Minor anomalies found: check the change report.


Brightway database written: ei_cutoff_3.12_remind-eu_SSP2-PkBudg1000_2030
Running core export checks...
Minor anomalies found: check the change report.


Brightway database written: ei_cutoff_3.12_remind-eu_SSP2-PkBudg1000_2040
Generate scenario report.
Report saved under /Users/timodiepers/Documents/Coding/bw_timex/notebooks/advanced/export/scenario_report.
Generate change report.
Report saved under /Users/timodiepers/Documents/Coding/bw_timex/notebooks/advanced/export/change reports/.


2026-08-26 08:47:43.677 | INFO     | bw_timex.scenario_builder:ensure_scenario_databases:424 - Built 3 background database(s).
2026-08-26 08:47:43.684 | INFO     | bw_timex.timex_lca:__init__:533 - Calculating base LCA...
/Users/timodiepers/Documents/Coding/bw_timex/.venv/lib/python3.12/site-packages/scikits/umfpack/umfpack.py:737: UmfpackWarning: (almost) singular matrix! (estimated cond. number: 8.06e+12)
  warnings.warn(msg, UmfpackWarning)
2026-08-26 08:47:44.888 | INFO     | bw_timex.timex_lca:__init__:578 - Foreground links into ei_cutoff_3.12_remind-eu_SSP2-PkBudg650_2020 which is not part of the requested background. Matching its processes into the requested vintages by (name, reference product, location).
2026-08-26 08:47:44.889 | INFO     | bw_timex.timex_lca:__init__:588 - Collecting node infos...
2026-08-26 08:47:44.946 | INFO     | bw_timex.timex_lca:__init__:603 - Loading node metadata from 5 database(s)...
2026-08-26 08:47:49.422 | INFO     | bw_timex.timex_lca:__init__:

In [11]:
comparison.summary[["label", "base_score", "dynamic_score", "runtime_s"]]

,label,base_score,dynamic_score,runtime_s
0,Pathway SSP2-PkBudg650,23029.209074,27329.168394,7.232549
1,Pathway SSP2-PkBudg1000,23029.209074,28883.301235,33.010445
